In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
%reload_ext autoreload

In [ ]:
# Cell 1 (only one needed now)
from src import (
    extract_logon_features,
    extract_http_features,
    extract_email_features,
    fuse_feature_matrices,
    InsiderThreatDetector,
)

In [ ]:
# Cell 2 — extraction

hist_logon = extract_logon_features("data/r4.2/logon.csv", "2010-01-01", "2010-12-31", dev_mode=True)
hist_http  = extract_http_features("data/r4.2/http.csv",   "2010-01-01", "2010-12-31", dev_mode=True)
hist_email = extract_email_features("data/r4.2/email.csv", "2010-01-01", "2010-12-31", dev_mode=True)

live_logon = extract_logon_features("data/r4.2/logon.csv", "2011-01-01", None, dev_mode=True)
live_http  = extract_http_features("data/r4.2/http.csv",   "2011-01-01", None, dev_mode=True)
live_email = extract_email_features("data/r4.2/email.csv", "2011-01-01", None, dev_mode=True)

In [ ]:
import pandas as pd
df = pd.read_parquet(hist_logon)
print(df.columns.tolist())  # should include 'total_event_buckets' — the H5 rename
print(df.index.names)        # should be ['user', 'activity_date'''

In [ ]:
fused_hist = fuse_feature_matrices(
    hist_logon, hist_http, hist_email,
    output_prefix="historical_baseline",
)
fused_live = fuse_feature_matrices(
    live_logon, live_http, live_email,
    output_prefix="live_test",
)

In [ ]:
df = pd.read_parquet(fused_hist)
print(df.shape)
print(df.columns.tolist())  # should be all 15 features in MASTER_FEATURE_SCHEMA

In [ ]:
import glob
import pandas as pd

hist_files = sorted(glob.glob("features/historical_baseline_*.parquet"))
live_files = sorted(glob.glob("features/live_test_*.parquet"))

print(f"Historical files found: {len(hist_files)}")
for f in hist_files:
    df = pd.read_parquet(f)
    print(f"  {f}: {len(df):,} rows")

print(f"\nLive files found: {len(live_files)}")
for f in live_files:
    df = pd.read_parquet(f)
    print(f"  {f}: {len(df):,} rows")

In [ ]:
from src import InsiderThreatDetector, fuse_feature_matrices

# Point variables at the parquets that already exist on disk
import glob
hist_files = sorted(glob.glob("features/historical_baseline_*.parquet"))
live_files = sorted(glob.glob("features/live_test_*.parquet"))

fused_hist = hist_files[-1] if hist_files else None  # latest one
fused_live = live_files[-1] if live_files else None

print(f"Historical: {fused_hist}")
print(f"Live: {fused_live}")

In [ ]:
# Cell 4 — training
detector = InsiderThreatDetector(mad_threshold=3.5, missing_data_tolerance=0.18)
pkl_path = detector.fit_baseline(fused_hist)
print(f"Trained model saved to: {pkl_path}")

In [ ]:
import joblib
d = joblib.load(pkl_path)
print("is_fitted:           ", d.is_fitted)
print("features:            ", list(d.baseline_stats.keys()))
print("iso_threshold:       ", d.iso_threshold)
print("has train_percentiles:", hasattr(d, "train_score_percentiles"))
print("model_version:       ", d.model_version)

In [ ]:
from src.detector import InsiderThreatDetector

detector2 = InsiderThreatDetector.load("iso_pipeline_v20260517_195228.pkl")
detector2.critical_flag_ratio = 0.10   # require 10% of features anomalous, not 26.7%
df_scores2 = detector2.predict_live_traffic("features\live_test_20260517_025131.parquet")

print(df_scores2["confirmed_threat"].sum())   # expect ~490
print(df_scores2["high_risk_review"].sum())   # still 0 until Problem 2 is fixed

In [ ]:
import pandas as pd

df_fused_live = pd.read_parquet("features/live_test_20260517_210025.parquet")

# Should show non-zero for off_hours_ratio and failed_login_ratio
print(df_fused_live[["off_hours_ratio", "failed_login_ratio"]].isnull().sum())
print(f"Total NaNs anywhere: {df_fused_live.isnull().sum().sum()}")

In [ ]:
import importlib
import src.fusion
importlib.reload(src.fusion)

from src.fusion import fuse_feature_matrices

In [ ]:
df_logon = pd.read_parquet("features/logon_features_20260517_205943.parquet")
df_http  = pd.read_parquet("features/http_features_20260517_205943.parquet")
df_email = pd.read_parquet("features/email_features_20260517_210008.parquet")

print("LOGON NaNs:", df_logon.isnull().sum().to_dict())
print("HTTP  NaNs:", df_http.isnull().sum().to_dict())
print("EMAIL NaNs:", df_email.isnull().sum().to_dict())

In [ ]:
import pandas as pd

# Read the fused parquet your last run produced
df_fused = pd.read_parquet("features/live_test_20260517_211953.parquet")
print(df_fused[["off_hours_ratio", "failed_login_ratio"]].isnull().sum())

In [ ]:
# Step 1 — force reload the fixed fusion module
import importlib, src.fusion
importlib.reload(src.fusion)
from src.fusion import fuse_feature_matrices

# Step 2 — re-fuse using the already-correct source parquets
# No need to re-extract — source parquets are fine
live_path = fuse_feature_matrices(
    logon_parquet_path = "features/logon_features_20260517_205943.parquet",
    http_parquet_path  = "features/http_features_20260517_205943.parquet",
    email_parquet_path = "features/email_features_20260517_210008.parquet",
    output_prefix      = "live_test",
)

# Step 3 — verify NaNs survived fusion this time
df_fused_new = pd.read_parquet(live_path)
print("off_hours_ratio NaNs: ", df_fused_new["off_hours_ratio"].isnull().sum())
print("failed_login_ratio NaNs:", df_fused_new["failed_login_ratio"].isnull().sum())
print("Total NaNs:", df_fused_new.isnull().sum().sum())

In [ ]:
import pandas as pd
from pathlib import Path
from src.fusion import fuse_feature_matrices
from src.detector import InsiderThreatDetector
from diagnostics import run_full_diagnostic

feature_dir = Path("features")

# ── Auto-select latest source parquets ────────────────────────────────────
latest_logon = max(feature_dir.glob("logon_features_*.parquet"), key=lambda p: p.stat().st_mtime)
latest_http  = max(feature_dir.glob("http_features_*.parquet"),  key=lambda p: p.stat().st_mtime)
latest_email = max(feature_dir.glob("email_features_*.parquet"), key=lambda p: p.stat().st_mtime)

print("── Source parquets selected ──────────────────────────────")
print(f"  Logon: {latest_logon.name}")
print(f"  HTTP:  {latest_http.name}")
print(f"  Email: {latest_email.name}")

# ── Verify NaNs exist in source parquets before fusion ───────────────────
print("\n── NaN check in source parquets ──────────────────────────")
for label, path in [("LOGON", latest_logon), ("HTTP", latest_http), ("EMAIL", latest_email)]:
    df = pd.read_parquet(path)
    print(f"  {label}: off_hours_ratio={df['off_hours_ratio'].isnull().sum():,}  "
          f"failed_login_ratio={df['failed_login_ratio'].isnull().sum():,}")

# ── Fuse ──────────────────────────────────────────────────────────────────
print("\n── Running fusion ────────────────────────────────────────")
live_path_new = fuse_feature_matrices(
    logon_parquet_path = str(latest_logon),
    http_parquet_path  = str(latest_http),
    email_parquet_path = str(latest_email),
    output_prefix      = "live_test_fixed",
)

# ── Verify NaNs survived fusion ───────────────────────────────────────────
df_fused = pd.read_parquet(live_path_new)
print("\n── NaN check in fused parquet ────────────────────────────")
print(f"  off_hours_ratio:    {df_fused['off_hours_ratio'].isnull().sum():,}")
print(f"  failed_login_ratio: {df_fused['failed_login_ratio'].isnull().sum():,}")
print(f"  Total NaNs:         {df_fused.isnull().sum().sum():,}")

# ── Only proceed if NaNs survived ─────────────────────────────────────────
total_nans = df_fused.isnull().sum().sum()
if total_nans == 0:
    print("\n  ✗ NaNs did not survive. Source parquets may still be stale.")
    print("  Re-run extract_logon/http/email_features() and use the new paths.")
else:
    print(f"\n  ✓ {total_nans:,} NaNs survived — proceeding to inference.")

    # ── Inference ─────────────────────────────────────────────────────────
    detector  = InsiderThreatDetector.load("iso_pipeline_v20260517_210132.pkl")
    df_scores = detector.predict_live_traffic(live_path_new)
    df_live   = InsiderThreatDetector._load_with_user_day_index(
                    live_path_new, context="diag"
                )
    run_full_diagnostic(detector, df_scores, df_live)

In [ ]:
from src.extraction import (
    extract_logon_features,
    extract_http_features,
    extract_email_features,
)
from src.fusion import fuse_feature_matrices
from src.detector import InsiderThreatDetector
from diagnostics import run_full_diagnostic
import pandas as pd

# ── Re-extract all three sources ──────────────────────────────────────────
print("── Extracting fresh source parquets ──────────────────────")

logon_path = extract_logon_features(
    user_input_path = "data/r4.2/logon.csv",
    start_date      = "2011-01-01",   # your live window start
    end_date        = None,   # your live window end
    output_prefix   = "live_logon",
    dev_mode        = True,
)

http_path = extract_http_features(
    user_input_path = "data/r4.2/http.csv",
    start_date      = "2011-01-01",
    end_date        = None,
    output_prefix   = "live_http",
    dev_mode        = True,
)

email_path = extract_email_features(
    user_input_path = "data/r4.2/email.csv",
    start_date      = "2011-01-01",
    end_date        = None,
    output_prefix   = "live_email",
    dev_mode        = True,
)

# ── Verify NaNs exist in freshly extracted parquets ───────────────────────
print("\n── NaN check in fresh source parquets ────────────────────")
for label, path in [("LOGON", logon_path), ("HTTP", http_path), ("EMAIL", email_path)]:
    df = pd.read_parquet(path)
    print(f"  {label}: off_hours_ratio={df['off_hours_ratio'].isnull().sum():,}  "
          f"failed_login_ratio={df['failed_login_ratio'].isnull().sum():,}")

# ── Fuse ──────────────────────────────────────────────────────────────────
print("\n── Fusing ────────────────────────────────────────────────")
live_path = fuse_feature_matrices(
    logon_parquet_path = logon_path,
    http_parquet_path  = http_path,
    email_parquet_path = email_path,
    output_prefix      = "live_test_fixed",
)

# ── Verify NaNs survived fusion ───────────────────────────────────────────
df_fused = pd.read_parquet(live_path)
print("\n── NaN check in fused parquet ────────────────────────────")
print(f"  off_hours_ratio:    {df_fused['off_hours_ratio'].isnull().sum():,}")
print(f"  failed_login_ratio: {df_fused['failed_login_ratio'].isnull().sum():,}")
print(f"  Total NaNs:         {df_fused.isnull().sum().sum():,}")

total_nans = df_fused.isnull().sum().sum()
if total_nans == 0:
    print("\n  ✗ Still zero NaNs — share the output of:")
    print("    pd.read_parquet(logon_path)[['off_hours_ratio']].isnull().sum()")
    print("  to confirm align_to_master_schema is running correctly.")
else:
    print(f"\n  ✓ {total_nans:,} NaNs survived. Proceeding to inference.")

    detector  = InsiderThreatDetector.load("iso_pipeline_v20260517_210132.pkl")
    df_scores = detector.predict_live_traffic(live_path)
    df_live   = InsiderThreatDetector._load_with_user_day_index(
                    live_path, context="diag"
                )
    run_full_diagnostic(detector, df_scores, df_live)

In [ ]:
import pandas as pd

df_logon_raw = pd.read_parquet(logon_path)
print("Columns in fresh logon parquet:")
print(df_logon_raw.columns.tolist())
print(f"\nShape: {df_logon_raw.shape}")
print(f"\noff_hours_ratio NaNs: {df_logon_raw['off_hours_ratio'].isnull().sum()}")
print(f"url_visits NaNs:      {df_logon_raw['url_visits'].isnull().sum()}")
print(f"\nFirst 3 rows:")
print(df_logon_raw.head(3))


In [ ]:
import pandas as pd
from pathlib import Path

latest_logon = max(Path("features").glob("live_logon_*.parquet"), key=lambda p: p.stat().st_mtime)
latest_http  = max(Path("features").glob("live_http_*.parquet"),  key=lambda p: p.stat().st_mtime)
latest_email = max(Path("features").glob("live_email_*.parquet"), key=lambda p: p.stat().st_mtime)

df_logon = pd.read_parquet(latest_logon)
df_http  = pd.read_parquet(latest_http)
df_email = pd.read_parquet(latest_email)

logon_idx = set(df_logon.index)
http_idx  = set(df_http.index)
email_idx = set(df_email.index)

print(f"Logon user-days:  {len(logon_idx):,}")
print(f"HTTP  user-days:  {len(http_idx):,}")
print(f"Email user-days:  {len(email_idx):,}")

print(f"\nHTTP  rows with NO logon record:  {len(http_idx  - logon_idx):,}")
print(f"Email rows with NO logon record:  {len(email_idx - logon_idx):,}")
print(f"Logon rows with NO HTTP record:   {len(logon_idx - http_idx):,}")
print(f"Logon rows with NO email record:  {len(logon_idx - email_idx):,}")

In [ ]:
import src.fusion, inspect
print(inspect.getsourcefile(src.fusion))
print("fillna(0) in source:", "df_fused.fillna(0)" in inspect.getsource(src.fusion))
print("COUNT_COLS in source:", "COUNT_COLS" in inspect.getsource(src.fusion))

In [ ]:
# Step 1 — delete pycache
import shutil, pathlib
for p in pathlib.Path("src").rglob("__pycache__"):
    shutil.rmtree(p)
    print(f"Deleted: {p}")

In [ ]:
import importlib 
from src.fusion import schema

# Force Python to ignore __pycache__ and always read .py directly
import importlib.util
spec   = importlib.util.spec_from_file_location(
    "src.fusion",
    "src/fusion.py",           # adjust path if needed
    submodule_search_locations=[]
)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
sys.modules["src.fusion"] = module

# Verify
import inspect
source = inspect.getsource(module)
print("fillna(0) present:", "df_fused.fillna(0)" in source)  # must be False
print("COUNT_COLS present:", "COUNT_COLS" in source)           # must be True

# Use directly
fuse_feature_matrices = module.fuse_feature_matrices